Welcome! Today you will expand the moneymodel to learn a variety of features that an Agent Based Model can do.
Let's start:

### Movement

*As people walk this way again and again, a path appears.*
-Lu Xun


The first expansion block will of course involve movement.

After def__init__ in class moneyagent add these two blocks one after the other by replacing the lines of def exchange(self):

The first block defines the ability of an agent to move.

The second one defines the ability of an agent to give money to a random other agent that is in the same cell as them.

In [ ]:
    def move(self):
        #gets the neighbourhood around this agent as options to move to
        possible_steps = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False
        )
        new_position = self.random.choice(possible_steps)
        self.model.grid.move_agent(self, new_position)

In [ ]:
    def give_money(self):
        #verify agent has some wealth
        if self.wealth > 0:
            cellmates = self.model.grid.get_cell_list_contents([self.pos])
            #ensure agent is not giving money to itself
            cellmates.pop(cellmates.index(self))
            if len(cellmates) > 0:
                other_agent = self.random.choice(cellmates)
                other_agent.wealth += 1
                self.wealth -= 1

After self.num_agents in class MoneyModel add these two blocks:

The first one produces the grid.

The second one creates and places the agents on a random spot on the grid

In [ ]:
        self.grid = mesa.space.MultiGrid(width, height, True)

In [ ]:
        #create agents
        agents = MoneyAgent.create_agents(model=self, n=n)
        #create x and y coordinates for agents
        x = self.rng.integers(0, self.grid.width, size=(n,))
        y = self.rng.integers(0, self.grid.height, size=(n,))
        for a, i, j in zip(agents, x, y):
            #add the agent to a random grid cell
            self.grid.place_agent(a, (i, j))

Don't forget to replace self.agents.shuffle_do("exchange") in the final step function to:

        self.agents.shuffle_do("move")
        self.agents.do("give_money")


Let's see where they end up at the end of the simulation, replace this at model setting of the run_moneymodel.ipynb file:

In [ ]:
model = MoneyModel(80, 10, 10)
for _ in range(100):
    model.step()
    
    agent_counts = np.zeros((model.grid.width, model.grid.height))
for cell_content, (x, y) in model.grid.coord_iter():
    agent_count = len(cell_content)
    agent_counts [x] [y] = agent_count
#plot using seaborn, with a visual size of 5x5
g = sns.heatmap(agent_counts, cmap="viridis", annot=True, cbar=False, square=True)
g.figure.set_size_inches(5, 5)
g.set(title="number of agents on each cell of the grid");
plt.show()

Awesome! Now we have agents that move around and give each other money. A wonderful world.


.



### Gini and Data Collection

*Equality seems to be just and it is, but not to everyone, only to those who are equal to begin with. And so, inequality seems to be just, and, indeed, it is, but not for all people, only to those who are not equal.*
-Aristotle

Of course our current code only shows us the last instance of wealth across our agents. And as good social scientists we are interested in understanding how things change over time. In this case how inequality changes over time. 

For this we will use the Gini index.


Right after importing libraries let’s add this block to define how we compute the gini index:

In [ ]:
def compute_gini(model):
    #making a list of all agent wealths
    agent_wealths = [agent.wealth for agent in model.agents]
    #defining and sorting from smallest to biggest
    x = sorted(agent_wealths)
    #defining how many agents we have
    n = model.num_agents
    #formulating the wealth value to calculate gini index
    B = sum(xi * (n - 1) for i, xi in enumerate(x)) / (n * sum(x))
    return 1 + (1 / n) - 2 * B


After self.grid in class MoneyModel let's add this block to collect the gini index for that step into a data collector:

In [ ]:
        #datacollector let's us collect and save data at every step of the model
        self.datacollector = mesa.DataCollector(
            model_reporters={"Gini": compute_gini}, agent_reporters={"Wealth": "wealth"}
        )


Don’t forget to add the data collection to our step:


In [ ]:

        self.datacollector.collect(self)

Now let's populate a graph showing the change in gini index in the run_money_model.ipynb file:

In [ ]:
gini = model.datacollector.get_model_vars_dataframe()
#plot the Gini coefficient over time
g = sns.lineplot(data=gini)
g.set(title="Gini Coefficient over Time", ylabel="Gini Coefficient")
plt.show()

Great, we can see how the gini index changes over time in our simulation.
Feel free to just run:
gini
As a separate cell to see what the data looks like.

Additional content: The next lines of code can be run separately in the same file to just see some interesting visualizations of data if interested, first block is needed for the rest:
1) Agent wealth data
2) Distribution of wealth at the end of the simulation
3) A single agent's wealth change
4) Comparing 3 agents
5) Looking at Wealth line plots



In [ ]:
agent_wealth = model.datacollector.get_agent_vars_dataframe()
agent_wealth.head()

In [ ]:
#distribtion of wealth at the end of the simulation
last_step = agent_wealth.index.get_level_values("Step").max()
end_wealth = agent_wealth.xs(last_step, level="Step")["Wealth"]
#create a histogram of wealth at the last step
g = sns.histplot(end_wealth, discrete=True)
g.set(
    title="Distribution of wealth at the end of simulation",
    xlabel="Wealth",
    ylabel="number of agents",
);

In [ ]:
#get the wealth of agent 7 over time
one_agent_wealth = agent_wealth.xs(7, level="AgentID")

#plot the wealth of agent 7 over time
g = sns.lineplot(data=one_agent_wealth, x="Step", y="Wealth")
g.set(title="Wealth of agent 7 over time");

In [ ]:
agent_list = [3, 14, 25]
#get the wealth of multiple agents over time
multiple_agents_wealth = agent_wealth[
    agent_wealth.index.get_level_values("AgentID").isin(agent_list)
]
#plot the wealth of multiple agents over time
g = sns.lineplot(data=multiple_agents_wealth, x="Step", y="Wealth", hue="AgentID")
g.set(title="Wealth of agents 3, 14 and 25 over time");

In [ ]:
#transform the data to a long format
agent_wealth_long = agent_wealth.T.unstack().reset_index()
agent_wealth_long.columns = ["Step", "AgentID", "Variable", "Value"]
agent_wealth_long.head(3)

#plot the average wealth over time
g = sns.lineplot(data=agent_wealth_long, x="Step", y="Value", errorbar=("ci", 95))
g.set(title="Average wealth over time")

.

### Visualization

*There is only one way to see things, until someone shows us how to look at them with different eyes* - Pablo Picasso

Great we can see the outcome of our model. But what is a model for if we can't see it in action. If we have not seen the agents interact then it is hard for us to realize what exactly is happening behind the screen. So, let's visualize the process.


To save time I have added the full visualization ipynb as run_visualization.ipynb to make sure you have all the libraries make sure you do:

source .venv/bin/activate
pip3 install mesa altair
pip3 install mesa solara

For windows remove the 3

Run through the model and see how it looks for yourself.

How does the model change if you change the number of agents? If you change the grid size?
Do you think urban density has an impact on inequality?


.

### Batching
*No sociologist should think himself too good, even in his old age, to make tens of thousands of quite trivial computations in his head and perhaps for months at a time.* - Max Weber

Well what if it just so happens that having 50 agents on a 10 by 10 grid always leads to inequality? What if we got lucky? 

This is where we'll apply batching in agent based modelling to prove that it wasn't a fluke. Going back to our question we will now batch our model so that it runs for differing amounts of agents.

To do so let's use this mesa batching command in our ipynb file:


In [ ]:
#change these to set the model parameters:
params = {"width": 10, "height": 10, "n": range(5, 100, 5)}

#input our model name and params into the batch_run method
results = mesa.batch_run(
    MoneyModel,
    parameters=params,
    iterations=5,
    max_steps=100,
    number_processes=1,
    data_collection_period=1,
    display_progress=True,
)


Now take a look at our data, what do you see?:

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.keys())

To simplify our dataset we will filter for only one agent since the gini index is the same for the whole population at each point in time. And then we can graph it against the number of agents

In [ ]:
#filter the results to only contain the data of one agent
#the Gini coefficient will be the same for the entire population at any time
results_filtered = results_df[(results_df.AgentID == 1) & (results_df.Step == 100)]
results_filtered[["iteration", "n", "Gini"]].reset_index(
    drop=True
).head()  # Create a scatter plot
g = sns.scatterplot(data=results_filtered, x="n", y="Gini")
g.set(
    xlabel="number of agents",
    ylabel="Gini coefficient",
    title="Gini coefficient vs. number of agents",
);

Now let's our point plots with error bars to see the difference between iterations.

In [ ]:
#create a point plot with error bars
g = sns.pointplot(data=results_filtered, x='n', y="Gini", linestyle="None")
g.figure.set_size_inches(8, 4)
g.set(
    xlabel="number of agents",
    ylabel="Gini coefficient",
    title="Gini coefficient vs. number of agents",
);

Okay so now for only 5 numbers of agents (5, 10, 20, 40, 80) but 25 iterations each to find where the numbers converge:

In [ ]:

params = {"seed": None, "width": 10, "height": 10, "n": [5, 10, 20, 40, 80]}

results_5s = mesa.batch_run(
    MoneyModel,
    parameters=params,
    iterations=25,
    max_steps=100,
    number_processes=1,
    data_collection_period=1,  # Need to collect every step
    display_progress=True,
)

results_5s_df = pd.DataFrame(results_5s)

In [ ]:

#create a lineplot with error bars
g = sns.lineplot(
    data=results_5s_df,
    x="Step",
    y="Gini",
    hue="n",
    errorbar=("ci", 95),
    palette="tab10",
)
g.figure.set_size_inches(8, 4)
plot_title = (
    "Gini coefficient for different population sizes\n"
    "(mean over 100 runs, with 95% confidence interval)"
)
g.set(title=plot_title, ylabel="Gini coefficient");

This was a good example of batching, you can do the same for other parameters like grid size or amount of starting money and see how it changes the outcomes.

### Conclusion

Together we have touched the following agent based modeling mechanics: running simple rules, adding movement, collecting data from each step, vizualizing and batching.